In [1]:
from pyspark.sql.functions import *
from pyspark.sql import *

Calculation started (calculation_id=7ec59f8f-0496-6dc8-2df2-d11002406ab9) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


# Import business.json

In [2]:
input_bucket = "s3://6242-project"
    
# Load Business Data
business_path = '/yelp_academic_dataset_business.json'
business_json = spark.read.json(input_bucket + business_path)
print("rows: ", business_json.count())

Calculation started (calculation_id=4ac59f8f-0ded-1c92-261c-6e721f4b9dfa) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
rows:  150346



In [3]:
business_json.printSchema()

Calculation started (calculation_id=88c59f8f-3e63-1c4e-a092-b3ba9fef740f) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: string (nullable = true)
 |    

In [4]:
print(business_json.first())

Calculation started (calculation_id=12c59f8f-5108-0949-f63c-96c3cd1300a6) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
Row(address='1616 Chapala St, Ste 2', attributes=Row(AcceptsInsurance=None, AgesAllowed=None, Alcohol=None, Ambience=None, BYOB=None, BYOBCorkage=None, BestNights=None, BikeParking=None, BusinessAcceptsBitcoin=None, BusinessAcceptsCreditCards=None, BusinessParking=None, ByAppointmentOnly='True', Caters=None, CoatCheck=None, Corkage=None, DietaryRestrictions=None, DogsAllowed=None, DriveThru=None, GoodForDancing=None, GoodForKids=None, GoodForMeal=None, HairSpecializesIn=None, HappyHour=None, HasTV=None, Music=None, NoiseLevel=None, Open24Hours=None, OutdoorSeating=None, RestaurantsAttire=None, RestaurantsCounterService=None, RestaurantsDelivery=None, RestaurantsGoodForGroups=None, RestaurantsPriceRange2=None, RestaurantsReservations=None, RestaurantsTableService=None, RestaurantsTakeOut=None, Smoking=None, WheelchairAccessible=None, WiFi=None), business_id='Pns2l4eNsfO8kk83dixA6A', categories='Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupunct

# Import review.json

In [5]:
# Load Review Data
review_path = '/yelp_academic_dataset_review.json'
review_json = spark.read.json(input_bucket + review_path)
print("rows: ", review_json.count())

Calculation started (calculation_id=74c59f8f-6462-1c79-fee5-4fc7c19d3189) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
rows:  6990280



In [6]:
review_json.printSchema()

Calculation started (calculation_id=d6c59f8f-9c32-797b-f5ae-207ee02e3e30) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
print(review_json.first())

Calculation started (calculation_id=7ec59f8f-ab01-0f90-bfe2-62d0d3bbf1f7) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
Row(business_id='XQfwVwDr-v0ZS3_CbbE5Xw', cool=0, date='2018-07-07 22:09:11', funny=0, review_id='KU_O5udG6zpxOg-VcAEodg', stars=3.0, text="If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. \n\nThe food is good, but it takes a very long time to come out. The waitstaff is very young, but usually pleasant. We have just had too many experiences where we spent way too long waiting. We usually opt for another diner or restaurant on the weekends, in order to be done quicker.", useful=0, user_id='mh_-eMZ6K5RLWhZyISBhwA')



# Select & clean columns from business and review

In [8]:
# Select only relevant columns from business_json
business = (
    business_json.select("business_id", "name", "city", "state", "latitude", "longitude", "stars", "review_count")
        .withColumnRenamed("business_id", "business_id_b")
    # replace double quotes in col('name') with single quotes
        .withColumn("name_clean", regexp_replace(col("name"), '"', "'")).drop("name")
)

Calculation started (calculation_id=bec59f90-f15a-6131-176b-e87c84d175e5) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


In [9]:
business.show(5)

Calculation started (calculation_id=42c59f90-fd17-65a6-8b8c-694ca661af9c) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
+--------------------+-------------+-----+----------+------------+-----+------------+--------------------+
|       business_id_b|         city|state|  latitude|   longitude|stars|review_count|          name_clean|
+--------------------+-------------+-----+----------+------------+-----+------------+--------------------+
|Pns2l4eNsfO8kk83d...|Santa Barbara|   CA|34.4266787|-119.7111968|  5.0|           7|Abby Rappoport, L...|
|mpf3x-BjTdTEA3yCZ...|       Affton|   MO| 38.551126|  -90.335695|  3.0|          15|       The UPS Store|
|tUFrWirKiKi_TAnsV...|       Tucson|   AZ| 32.223236| -110.880452|  3.5|          22|              Target|
|MTSW4McQd7CbVtyjq...| Philadelphia|   PA|39.9555052| -75.1555641|  4.0|          80|  St Honore Pastries|
|mWMc6_wTdE0EUBKIG...|   Green Lane|   PA|40.3381827| -75.4716585|  4.5|          13|Perkiomen Valley ...|
+--------------------+-------------+-----+----------+------------+-----+------------+--------------------+
only showing t

In [10]:
# Select only relevant columns from review_json
review = (
    review_json.select("review_id", "business_id", "stars", "date", "text")
        .withColumnRenamed("stars", "rating")
    # replace newline with space
        .withColumn("text_nolines", regexp_replace(col("text"), "[\n\r]", " ")).drop("text")
    # replace double quote with single quote
        .withColumn("text_clean", regexp_replace(col("text_nolines"), '"', "'")).drop("text_nolines")
)

Calculation started (calculation_id=0cc59f91-55be-ba83-9211-6605ad50afea) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


# Create small dataset for testing

In [11]:
first5_reviews = review.limit(5)
first5_reviews.show()

Calculation started (calculation_id=60c59f91-774b-e16d-bec1-b999af914163) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
+--------------------+--------------------+------+-------------------+--------------------+
|           review_id|         business_id|rating|               date|          text_clean|
+--------------------+--------------------+------+-------------------+--------------------+
|8H81chukQT03OySkD...|_pbx96FZ3eHJw-V_R...|   4.0|2014-07-24 12:26:21|Overall this plac...|
|HXp5PP4qS9R2MTsU0...|-e9MepGs8piOYdwuP...|   5.0|2015-08-05 02:59:54|Great, safe space...|
|vhtji9oev08UlGJzr...|PY9GRfzr4nTZeINf3...|   5.0|2011-03-17 00:38:46|We were marrried ...|
|esWX--kcr2qOBOFhO...|37BpNvlEAT6WVGsks...|   4.0|2014-03-28 22:32:10|Order a salad wit...|
|CnTgvZgwgYpV1R3Zr...|OUSHKQ9AaBNvX90fU...|   5.0|2017-05-11 13:01:08|Brittany did an a...|
+--------------------+--------------------+------+-------------------+--------------------+



# Complete merged dataset

In [15]:
data = (
    business.join(review, business["business_id_b"] == review["business_id"])
        .drop("business_id")
        .withColumnRenamed("business_id_b", "business_id")
        .withColumnRenamed("name_clean", "name")
        .withColumnRenamed("text_clean", "text")
        .select("business_id", "name", "city", "state", "latitude", "longitude", "stars", "review_count", "review_id", "rating", "date", "text")
)

Calculation started (calculation_id=9ac59f93-8fde-976f-ddd5-6b90bb62ceca) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


In [16]:
data.printSchema()

Calculation started (calculation_id=e0c59f93-aad5-102e-39f1-e5a865f8e142) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
root
 |-- business_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- stars: double (nullable = true)
 |-- review_count: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- date: string (nullable = true)
 |-- text: string (nullable = true)



In [17]:
data.show(5)

Calculation started (calculation_id=0ec59f93-dd4b-4ecd-a92d-2697e5d93075) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
+--------------------+--------------------+-----+-----+----------+-----------+-----+------------+--------------------+------+-------------------+--------------------+
|         business_id|                name| city|state|  latitude|  longitude|stars|review_count|           review_id|rating|               date|                text|
+--------------------+--------------------+-----+-----+----------+-----------+-----+------------+--------------------+------+-------------------+--------------------+
|--gJkxbsiSIwsQKbi...|Salon Lofts - Wes...|Tampa|   FL|27.9451223|-82.5210814|  5.0|           6|4SPOoOr1ZZWGm-1Om...|   5.0|2019-01-10 02:51:06|Amber is the best...|
|--gJkxbsiSIwsQKbi...|Salon Lofts - Wes...|Tampa|   FL|27.9451223|-82.5210814|  5.0|           6|CSlZvn9wPq6kIahbc...|   5.0|2019-04-18 18:34:40|Gina Marotti in L...|
|--gJkxbsiSIwsQKbi...|Salon Lofts - Wes...|Tampa|   FL|27.9451223|-82.5210814|  5.0|           6|gMGm7d8b8pXwi1Bzl...|   5.0|2018-12-10 21:36:

In [18]:
# Confirm number of rows in final dataset == 6990280
data.count()

Calculation started (calculation_id=80c59f94-6a61-2bc4-fc55-59da4ba6d802) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
6990280



# Write to bucket

In [19]:
bucket = 's3://6242-project/business_reviews_joined'
data.coalesce(1).write.csv(bucket, header = True, mode = 'overwrite')

Calculation started (calculation_id=72c59f94-ab94-5594-b90a-1a4fc2ab1b3c) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


# TESTING BELOW: Processing first 5 rows only

In [21]:
test = (
    business.join(first5_reviews, business["business_id_b"] == first5_reviews["business_id"])
        .drop("business_id")
        .withColumnRenamed("business_id_b", "business_id")
        .withColumnRenamed("name_clean", "name")
        .withColumnRenamed("text_clean", "text")
        .select("business_id", "name", "city", "state", "latitude", "longitude", "stars", "review_count", "review_id", "rating", "date", "text")
)

test = test.coalesce(1)

Calculation started (calculation_id=e6c59f96-68d3-d6ff-c924-f0235a22bfde) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


In [22]:
test.show()
#test.first()
#test.printSchema()

Calculation started (calculation_id=28c59f96-6e24-dd11-e3f9-5a5f9799d0a3) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
+--------------------+--------------------+-----------+-----+-------------+--------------+-----+------------+--------------------+------+-------------------+--------------------+
|         business_id|                name|       city|state|     latitude|     longitude|stars|review_count|           review_id|rating|               date|                text|
+--------------------+--------------------+-----------+-----+-------------+--------------+-----+------------+--------------------+------+-------------------+--------------------+
|PGv11T-KjPvB4xoRS...|      Sunrise Eatery|Zephyrhills|   FL|     28.24308|    -82.188305|  4.5|         281|BCDwoMOXechhbfVhL...|   5.0|2021-01-23 16:51:01|This was our firs...|
|whh_h_yJaF2kz-ieB...|         First Watch|     Tucson|   AZ|       32.251|    -110.89084|  4.5|         226|rrBt0eSsTgqv5c43N...|   5.0|2018-09-30 19:57:07|Love this place. ...|
|twuh803Q03U9X1YGn...|  Stella Of New Hope|   New Hope|   PA|   40.3631045|   -74.

In [23]:
# Write to s3:
test_bucket = 's3://6242-project/first5reviews'
test.write.csv(test_bucket, header = True, mode = 'overwrite')

Calculation started (calculation_id=7ac59f97-32cd-ffe8-f553-41f0c8b49bfb) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


In [24]:
# Load Test Data
input_bucket = "s3://6242-project"
# test_path file name will change every time you write, make sure to change path below accordingly
test_path ='/first5reviews/part-00000-10f321b3-ce1c-40bd-b8d4-8f46ccbe228e-c000.csv'
test_csv = spark.read.csv(input_bucket + test_path, header=True)
print("rows: ", test_csv.count()) # should have rows == 5

Calculation started (calculation_id=30c59f97-9de9-ffbc-db89-44fdfd995508) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
rows:  5



In [ ]:
test_csv.show()

Calculation started (calculation_id=24c59f97-b881-7a96-0120-edd90d514d24) in (session=b8c59f8e-f2bd-7941-cf2e-d0d3b1ac0a64). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s